# 👨‍⚖️ LLM-as-a-Judge: Automated Intent Evaluation

Welcome!
Manually grading thousands of AI logs is impossible in production. That is where **LLM-as-a-Judge** comes in. While this allows us to scale our testing without needing humans to read every single row, it introduces a new challenge: Trust. How do we know that the judge is right?

The answer lies in rigorous calibration. We need to align our LLM-Judge's evaluations with human expertise and expected outcomes.

## 📚 Technical Concepts Covered

### **1. Deterministic Inference**
Initializing the Judge with `temperature=0` to ensure the model is as deterministic and factual as possible
### **2. Schema Enforcement (Structured Output)**
Using **Pydantic** to force the LLM to return valid Boolean flags (TRUE/FALSE) instead of rambling text.
### **3. Batch Inference**
Scaling the logic to process a full dataset using Pandas.
### **4. Alignment**
Calculating **Precision**, **Recall**, and **F1-Scores** to measure how well our AI Judge agrees with Human Ground Truth.

Let's set up the lab. We need `dotenv` to handle our API keys securely and `pydantic` for **data validation**.

In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from typing import Literal
from prompts.llm_as_judge_prompt import VAGUE_UTTERANCES_FAILURE_PROMPT
from sklearn.metrics import classification_report, confusion_matrix 

import pandas as pd

In [2]:
load_dotenv(dotenv_path=".env", override=True)

True

![Diagram](./images/05_img.png)

## 1. Initializing the Judges
Time to meet the evaluators. In this architecture, the Judge usually needs to be a stronger model than the one being tested.

### 🧠 Code Context
* `temperature=0`: This is critical for evaluation. We want **Deterministic Inference** meaning if we run this test tomorrow, we want the exact same grade. We don't want the judge getting "creative."

In [3]:
llm_as_judge_1 = init_chat_model("gpt-4o", model_provider="openai",temperature=0)
llm_as_judge_2 = init_chat_model("gpt-5-mini", model_provider="openai",temperature=0) 

## 2. Schema Enforcement (Structured Output)
LLMs love to chat, but we can't calculate metrics on a paragraph of text. We need a clean `TRUE` or `FALSE`.
To achieve this, we use **Schema Enforcement**. We bind a Pydantic object to the model, forcing it to output valid JSON that matches our strict rules.

### 🧠 Code Context
* `JudgeOutputSchema`: The blueprint for the output.
* `label`: Constrained to `Literal["TRUE", "FALSE"]`.
* `reasoning`: We ask for this to enable **Chain-of-Thought (CoT)**. Forcing the model to explain *why* before it decides *true/false* significantly improves accuracy.

In [4]:
def llm_as_judge_eval(user_query, predicted_intent, llm_as_judge):
    class JudgeOutputSchema(BaseModel):
        """Structured output for Doctor intent classification"""

        label: Literal["TRUE", "FALSE"] = Field(
            description="TRUE if the model’s intent is misclassified because the user’s utterance is vague else FALSE"
        )

        reasoning: str = Field(
            description="Brief explanation of the reason for the classification"
        )

    llm_with_structured_output = llm_as_judge.with_structured_output(JudgeOutputSchema)
    template = PromptTemplate.from_template(VAGUE_UTTERANCES_FAILURE_PROMPT)

    chain = template | llm_with_structured_output

    result = chain.invoke({'user_query': user_query, 'predicted_intent': predicted_intent})

    return result

## 3. Unit Testing (Sanity Check)
Before we run the judge on the whole dataset, let's run a **Unit Test**. We will feed it one edge case to make sure our logic holds up.

### 🧠 Code Context
* **Input:** *"Clear my 3 PM"* (A classic vague query).
* **Expected Result:** The Judge should flag this as `TRUE` (Failure) because "Clear" is ambiguous.

In [5]:
user_query = 'Clear my 3 PM'
predicted_intent = 'UNSUPPORTED_REQUEST'
llm_as_judge = llm_as_judge_2
llm_as_judge_eval(user_query, predicted_intent, llm_as_judge)

JudgeOutputSchema(label='FALSE', reasoning='The user asked to "Clear my 3 PM" — a clear request to remove/cancel the 3 PM appointment(s), which maps to CANCEL_APPOINTMENTS. The time is specified, so the utterance is not vague; the model incorrectly labeled it UNSUPPORTED_REQUEST for another reason.')

# Judge Alignment Eval

## 4. Dataset Ingestion
We load the **Evaluation Dataset** (Golden Dataset). This CSV contains real historical data:
1.  `USER_INPUT`: The raw query from the user.
2.  `final_intent`: The intent predicted by our chatbot.
3.  `Vague_Utterance_Failure`: The **Human Ground Truth** label. We will use this to grade the Judge.

### 🧠 Code Context
* `pd.read_csv`: Loads the data into a Pandas DataFrame for efficient batch processing.

In [9]:
import pandas as pd

dataset_path = "data/llm_as_judge_dataset.csv"
# output_path = "judge_alignment_1.csv"
output_path = "data/judge_alignment_2.csv"

df = pd.read_csv(dataset_path)

In [10]:
df.head()

,TEST_ID,USER_INPUT,TARGET_INTENT,final_intent,final_reasoning,intent_match,Human Validator Comments (Open Codes),Failure Categories,Vague_Utterance_Failure
0,TEST_009,How many patients did I see last month?,UNSUPPORTED_REQUEST,IRRELEVANT_QUERY,The user's request is about historical data an...,fail,It is definitely a query that doctors might as...,Unsupported_vs_Irrelevant,False
1,TEST_016,Cancel my 10:00 - 10:30 patient Monday,CANCEL_APPOINTMENTS,UNSUPPORTED_REQUEST,The request is to cancel a specific appointmen...,fail,The system is capable of supporting canceling ...,False_Unsupported,False
2,TEST_031,I am busy today,BLOCK_SLOTS,UNSUPPORTED_REQUEST,The user's statement 'I am busy today' does no...,fail,It does sound like the intent here is to block...,Vague_Utterances,True
3,TEST_032,I'm not working *this* Saturday.,BLOCK_SLOTS,UNBLOCK_SLOTS,The user is indicating that they are not worki...,fail,It is a block slot intent. But the model is ma...,Negation_Error,False
4,TEST_034,Free up this weekend,UPDATE_SPECIAL_SLOTS,UNBLOCK_SLOTS,The user is requesting to make time slots avai...,fail,This is not an unblocked slot because unblocke...,Special_Slots_vs_Unblock,False


## 5. Batch Inference Pipeline
We now scale the process by running the `llm_as_judge_eval` function across every row in the dataset.

### 🧠 Code Context
* **Model Selection**: We assign `llm_as_judge_2` (the smaller model) to be the active judge for this run.
* `df.apply(...)`: This vectorizes the operation, sending each row to the LLM API.
* **Latency Warning**: Depending on dataset size, this step may take time as it requires an API call for every single row.

In [ ]:
# llm_as_judge = llm_as_judge_1 #runs in 20 mins
llm_as_judge = llm_as_judge_2 #runs in 35 mins

def apply_llm_judge(row, llm_as_judge):
    """Apply LLM judge evaluation to a single row."""
    result = llm_as_judge_eval(
        user_query=row['USER_INPUT'],
        predicted_intent=row['final_intent'],
        llm_as_judge=llm_as_judge
    )
    return pd.Series({
        'LLM_Vague_Utterance_Failure': result.label,
        'LLM_Judge_reasoning': result.reasoning
    })

# Apply to each row
df[['LLM_Vague_Utterance_Failure', 'LLM_Judge_reasoning']] = df.apply(
    lambda row: apply_llm_judge(row, llm_as_judge), 
    axis=1
)

In [36]:
df.head()

,TEST_ID,USER_INPUT,TARGET_INTENT,final_intent,final_reasoning,intent_match,Human Validator Comments (Open Codes),Failure Categories,Vague_Utterance_Failure,LLM_Vague_Utterance_Failure,LLM_Judge_reasoning
0,TEST_009,How many patients did I see last month?,UNSUPPORTED_REQUEST,IRRELEVANT_QUERY,The user's request is about historical data an...,fail,It is definitely a query that doctors might as...,Unsupported_vs_Irrelevant,False,FALSE,The user clearly asks for a historical count o...
1,TEST_016,Cancel my 10:00 - 10:30 patient Monday,CANCEL_APPOINTMENTS,UNSUPPORTED_REQUEST,The request is to cancel a specific appointmen...,fail,The system is capable of supporting canceling ...,False_Unsupported,False,FALSE,The correct intent is CANCEL_APPOINTMENTS — th...
2,TEST_031,I am busy today,BLOCK_SLOTS,UNSUPPORTED_REQUEST,The user's statement 'I am busy today' does no...,fail,It does sound like the intent here is to block...,Vague_Utterances,True,TRUE,User likely meant to block their calendar for ...
3,TEST_032,I'm not working *this* Saturday.,BLOCK_SLOTS,UNBLOCK_SLOTS,The user is indicating that they are not worki...,fail,It is a block slot intent. But the model is ma...,Negation_Error,False,FALSE,Correct intent is BLOCK_SLOTS (doctor unavaila...
4,TEST_034,Free up this weekend,UPDATE_SPECIAL_SLOTS,UNBLOCK_SLOTS,The user is requesting to make time slots avai...,fail,This is not an unblocked slot because unblocke...,Special_Slots_vs_Unblock,False,TRUE,"The phrase ""Free up this weekend"" is ambiguous..."


In [ ]:
# Save results to CSV
df.to_csv(output_path, index=False)


# Alignment

In [6]:
df = pd.read_csv('data/judge_alignment_1.csv')

## 6. Dataset Balance Analysis
Before analyzing performance, we must understand our data distribution. We check the balance of `TRUE` vs `FALSE` labels in the Human Ground Truth.

### 🧠 Why this matters?
* **Class Imbalance**: In this dataset, ~93% of rows are `False` (Not a failure) and only ~7% are `True` (Failures).
* This extreme imbalance means a "dumb" model could just guess "False" every time and still get 93% accuracy. This is why we need deeper metrics like **Precision** and **Recall** in the next steps.

In [7]:
class_distribution = (
    df["Vague_Utterance_Failure"].value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
    .assign(percentage=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    .sort_values("percentage", ascending = False)
)
class_distribution

,intent,count,percentage
0,False,280,93.33
1,True,20,6.67


## 7. Confusion Matrix Analysis
We generate a **Confusion Matrix** to visualize the Judge's performance against the Human Ground Truth.

### 🧠 Interpreting the Matrix
* **True Negatives (TN)**: Judge said "No Failure," Human said "No Failure" (Correct).
* **False Positives (FP)**: Judge said "Failure," Human said "No Failure" (False Alarm).
* **False Negatives (FN)**: Judge said "No Failure," Human said "Failure" (Missed Detection).
* **True Positives (TP)**: Judge said "Failure," Human said "Failure" (Correct Catch).

In [8]:
# Create confusion matrix with labels
cm = confusion_matrix(
    df["Vague_Utterance_Failure"].astype(str).str.lower(),
    df["LLM_Vague_Utterance_Failure"].astype(str).str.lower()
)

# Get unique labels in sorted order
labels = sorted(df["Vague_Utterance_Failure"].astype(str).str.lower().unique())

# Create a readable DataFrame
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {label}" for label in labels],
    columns=[f"Predicted: {label}" for label in labels]
)

print("Confusion Matrix:")
print("=" * 60)
print(cm_df)
print("\n" + "=" * 60)
print(f"Total samples: {cm.sum()}")
print(f"Correct predictions: {cm.diagonal().sum()}")
print(f"Accuracy: {(cm.diagonal().sum() / cm.sum() * 100):.2f}%")

Confusion Matrix:
               Predicted: false  Predicted: true
Actual: false               238               42
Actual: true                 11                9

Total samples: 300
Correct predictions: 247
Accuracy: 82.33%


## 8. Alignment Metrics (Precision & Recall)
This is the final "Report Card" for our AI Judge. We use `classification_report` to calculate the key metrics:

### 🧠 Metric Definitions
* **Precision (The "Trust" Metric)**: When the Judge claims it found a failure, how often is it right?
    * *Low Precision means too many False Alarms.*
* **Recall (The "Safety" Metric)**: Out of all the actual failures in the dataset, how many did the Judge catch?
    * *Low Recall means the Judge is missing actual bugs.*
* **F1-Score**: The harmonic mean of Precision and Recall. This gives us a single number to evaluate the Judge's reliability.

In [9]:
#Precision/Recall per class

class_report = classification_report(
    df["Vague_Utterance_Failure"].astype(str).str.lower(),
    df["LLM_Vague_Utterance_Failure"].astype(str).str.lower(),
    output_dict=True,
    zero_division=0,
)

pd.DataFrame(class_report).T

,precision,recall,f1-score,support
false,0.955823,0.850000,0.899811,280.000000
true,0.176471,0.450000,0.253521,20.000000
accuracy,0.823333,0.823333,0.823333,0.823333
macro avg,0.566147,0.650000,0.576666,300.000000
weighted avg,0.903866,0.823333,0.856725,300.000000


## 9. Comparing a Second Judge
We saved results for a second judge model (`llm_as_judge_2`) earlier. Here we reload those results and run the **same** confusion matrix and classification report on them.

### 🧠 Why this matters?
This lets us **A/B compare two judges** on identical data — do they agree with humans equally well? Comparing alignment scores across models is how you choose which judge to trust (and whether a cheaper model is good enough).

In [10]:
df = pd.read_csv('data/judge_alignment_2.csv')

In [11]:
# Create confusion matrix with labels
cm = confusion_matrix(
    df["Vague_Utterance_Failure"].astype(str).str.lower(),
    df["LLM_Vague_Utterance_Failure"].astype(str).str.lower()
)

# Get unique labels in sorted order
labels = sorted(df["Vague_Utterance_Failure"].astype(str).str.lower().unique())

# Create a readable DataFrame
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {label}" for label in labels],
    columns=[f"Predicted: {label}" for label in labels]
)

print("Confusion Matrix:")
print("=" * 60)
print(cm_df)
print("\n" + "=" * 60)
print(f"Total samples: {cm.sum()}")
print(f"Correct predictions: {cm.diagonal().sum()}")
print(f"Accuracy: {(cm.diagonal().sum() / cm.sum() * 100):.2f}%")

Confusion Matrix:
               Predicted: false  Predicted: true
Actual: false               257               23
Actual: true                 13                7

Total samples: 300
Correct predictions: 264
Accuracy: 88.00%


In [12]:
#Precision/Recall per class

class_report = classification_report(
    df["Vague_Utterance_Failure"].astype(str).str.lower(),
    df["LLM_Vague_Utterance_Failure"].astype(str).str.lower(),
    output_dict=True,
    zero_division=0,
)

pd.DataFrame(class_report).T

,precision,recall,f1-score,support
false,0.951852,0.917857,0.934545,280.00
true,0.233333,0.350000,0.280000,20.00
accuracy,0.880000,0.880000,0.880000,0.88
macro avg,0.592593,0.633929,0.607273,300.00
weighted avg,0.903951,0.880000,0.890909,300.00


### 🚀 Next Steps
Now that you have a working LLM Judge alignment pipeline, here are a few things you can consider:
1.  **Model Selection:** Run an A/B test between different judge models (Steps 5 & 9) to see if the smaller model provides enough accuracy at a lower cost.
2.  **Prompt Refinement:** If your **Recall** is low, try adding "Few-Shot Examples" to the judge prompt used in Step 2.

---

### 📚 Technical Resources
* **[LangSmith Evaluation](https://docs.smith.langchain.com/evaluation)**: Enterprise-grade tracing and evaluation for LLM apps.
* **[RAGas Framework](https://docs.ragas.io/en/stable/)**: A specialized framework for evaluating RAG pipelines.
* **[Scikit-Learn Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)**: Deep dive into Precision, Recall, and F1.
* **[Pydantic Docs](https://docs.pydantic.dev/latest/)**: Learn more about strict schema validation.